In [0]:
%pip install --upgrade kagglehub

In [0]:
import os
from pathlib import Path
import kagglehub
from pyspark.sql.functions import current_timestamp, lit

In [0]:
KAGGLE_DATASET = "lunthu/steam-monthly-average-players"
VOLUME_PATH = "/Volumes/steam_game_intelligence/bronze/kaggle_data"

CATALOG = "steam_game_intelligence"
SCHEMA = "bronze"
TABLE_NAMES = "steamcharts_2025_history_data"

SECRET_SCOPE = "kaggle"
SECRET_KEY = "kaggleAPIKey"

In [0]:
KAGGLE_API_TOKEN = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key=SECRET_KEY
)

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

print("Kaggle authentication configured.")

In [0]:
dataset_path = kagglehub.dataset_download(
    KAGGLE_DATASET,
    output_dir=VOLUME_PATH,
    force_download=True
)

print(f"Dataset downloaded to: {dataset_path}")

In [0]:
from pathlib import Path

files = [
    file
    for file in Path(VOLUME_PATH).rglob("*")
    if file.is_file()
]

for file in files:
    print(file)

In [0]:
steamcharts_2025_file = next(
    (
        file
        for file in files
        if file.name.lower() == "steamcharts.csv"
    ),
    None
)


if steamcharts_2025_file is None:
    raise FileNotFoundError(
        "Could not find 'steamcharts.csv' in the Kaggle dataset."
    )

print(f"Reading file: {steamcharts_2025_file}")

In [0]:
steamcharts_2025_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .load(str(steamcharts_2025_file))
)

In [0]:
steamcharts_2025_df = (
    steamcharts_2025_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn(
        "_source",
        lit("https://www.kaggle.com/datasets/lunthu/steam-monthly-average-players")
    )
)

In [0]:
print("Schema:")
steamcharts_2025_df.printSchema()

print(f"Rows: {steamcharts_2025_df.count()}")
print(f"Columns: {len(steamcharts_2025_df.columns)}")

display(steamcharts_2025_df.limit(20))

In [0]:
STEAMCHARTS_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAMES}"

(
    steamcharts_2025_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(STEAMCHARTS_TABLE)
)

print(f"Table created successfully:")
print(STEAMCHARTS_TABLE)